In [ ]:
# Importing the necessary packages
import numpy as np                                  # "Scientific computing"
import scipy.stats as stats                         # Statistical tests

import pandas as pd                                 # Data Frame
from pandas.api.types import CategoricalDtype

import matplotlib.pyplot as plt                     # Basic visualisation
from statsmodels.graphics.mosaicplot import mosaic  # Mosaic diagram
import seaborn as sns                               # Advanced data visualisation

In [ ]:
# Read dataset + data preparation
rlanders = pd.read_csv('https://raw.githubusercontent.com/HoGentTIN/dsai-en-labs/main/data/rlanders.csv').set_index(['ID'])
rlanders.Gender = rlanders.Gender.astype('category')
likert_scale = CategoricalDtype(categories=[1,2,3,4,5], ordered=True)
rlanders.Survey = rlanders.Survey.astype(likert_scale)
# rlanders.info()
# rlanders.Survey.dtype

# Chi squared (qualitative, qualitative)

In [ ]:
pd.crosstab(rlanders.Survey, rlanders.Gender, margins=True)
observed = pd.crosstab(rlanders.Survey, rlanders.Gender)
row_sums = observed.sum(axis=1)
col_sums = observed.sum()
n = row_sums.sum()
expected = np.outer(row_sums, col_sums) / n
exp_row_sums = np.sum(expected, axis=1)
exp_col_sums = np.sum(expected, axis=0)
alpha = 0.05

print(f'Row totals   : {exp_row_sums}')
print(f'Column totals: {exp_col_sums}')
print(f'Observations : {exp_col_sums.sum()}')

diffs = (expected - observed)**2 / expected
chi_squared = diffs.values.sum()

dimensions = observed.shape
dof = (dimensions[0]-1) * (dimensions[1]-1)

observed = pd.crosstab(rlanders.Survey, rlanders.Gender)
chi2, p, df, expected = stats.chi2_contingency(observed)
g = stats.chi2.isf(alpha, df = dof)
p = stats.chi2.sf(chi_squared, df=dof)
print(f"χ²                ≈ {chi_squared:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"Critical value    ≃ {g:.3f}")
print(f"p-value           : {p:.4f}")


## Stappenplan

> stap 1:  
> H_0: er is geen verband tussen x en y  
> H_1: er is wel een verband tussen x en y

In [ ]:
# stap 2
alpha = 0.05
# stap 3 -> teststatistiek = Zowel onafhankelijke als de afhankelijke verandelijke zijn kwalitatieve variabelen -> chi2
# stap 4 -> p + g berekenen
chi2, p, dof, expected = stats.chi2_contingency(observed)
g = stats.chi2.isf(alpha, df = dof)
print(f"χ²                ≈ {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"Critical value    ≃ {g:.3f}")
print(f"p-value           : {p:.4f}")

> stap 5:  
> $p$ $\gt$ $\alpha$ en $g$ $\gt$ $\chi$²  
> Er is niet voldoende aanwijzing om $H_0$ te verwerpen -> correcte sample  
> $p$ $\lt$ $\alpha$ en $g$ $\lt$ $\chi$²  
> Er is voldoende aanwijzing om $H_0$ te verwerpen -> correcte sample

The closer $\chi^2$ is to 0, the closer the observed values are to the expected values, and thus the weaker the association between the two variables. The larger $\chi^2$, the stronger the association.

# Cramer's V

In [ ]:
cramers_v = np.sqrt(chi_squared / ((min(observed.shape) - 1) * n))
stats.contingency.association(observed, method='cramer')

To draw a conclusion from this figure, compare it with the values in the table below:

| Cramér's V | Interpretation          |
| :---:      | :---                    |
| 0          | No association          |
| 0.1        | Weak association        |
| 0.25       | Moderate association    |
| 0.50       | Strong association      |
| 0.75       | Very strong association |
| 1          | Complete association    |

# Goodness-of-fit

In [ ]:
# Age groups=           18-25  26-35  36-45  46-55  56+
observed =   np.array([   75,    98,   127,    73,  27])
expected_p = np.array([  .17,   .23,   .35,   .17, .08])
alpha = 0.05               # Significance level
n = sum(observed)          # Sample size
k = len(observed)          # Number of categories
dof = k - 1                # Degrees of freedom
expected = expected_p * n  # Expected absolute frequencies in the sample
g = stats.chi2.isf(alpha, df=dof)  # Critical value

# Goodness-of-fit-test in Python:
chi2, p = stats.chisquare(f_obs=observed, f_exp=expected)

print("Significance level  ⍺ = %.2f" % alpha)
print("Sample size         n = %d" % n)
print("k = %d; df = %d" % (k, dof))
print("Chi-squared        χ² = %.4f" % chi2)
print("Critical value      g = %.4f" % g)
print("p-value             p = %.4f" % p)

We can see that $\chi^2$ in the sample is left of the critical value, so within the area of acceptance. Therefore, we cannot reject the null hypothesis and can conclude that the sample is representative for the population, at least w.r.t. the age groups.

# Standardized residuals

In [ ]:
families = pd.DataFrame(
    np.array(
        [[0,  58],
         [1, 149],
         [2, 305],
         [3, 303],
         [4, 162],
         [5,  45]]),
    columns=['num_boys', "observed"])
families.set_index(['num_boys'])
n = families.observed.sum() # sample size

from scipy.special import binom # binomial-function

# probability for a boy
prob_boy = .5
# Add new colum to the data frame for the expected percentages
families['expected_p'] = binom(5, families.num_boys) * prob_boy**families.num_boys * prob_boy**(5-families.num_boys)
# Expected absolute frequencies in the sample:
families['expected'] = families['expected_p'] * n

alpha=0.01                         # significance level
dof=len(families)-1                # degrees of freedom
g = stats.chi2.isf(alpha, df=dof)  # Critical value
# Perform Chi-squared test, calculate χ² and p
chi2, p = stats.chisquare(f_obs=families.observed, f_exp=families.expected)

print("Chi-squared   χ² = %.4f" % chi2)
print("Critical value g = %.4f" % g)
print("p-value        p = %f"   % p)

families['stdres'] = (families.observed - families.expected) / np.sqrt(families.expected * (1 - families.expected_p))

As long as $r_i \in [-2, 2]$, we consider the differences to be random sampling errors. A value $r_i < -2$ indicates underrepresentation of this category, $r_i > 2$ indicates overrepresentation.

# Cochran's rule

A chi-squared test can only give good results if you have enough observations in each category. The statistician Cochran (1954) formulated a rule of thumb to determine what exactly *enough* is on contingency tables larger than 2x2:

1. All expected values must be at least 1
2. At most 20% of the expected values may be smaller than 5

Consequently, if a contingency table does *not* meet these criteria, the results of the chi-squared test may not be reliable!